In [2]:
!pip install xgboost

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.5/69.5 MB 6.9 MB/s eta 0:00:11
    --------------------------------------- 1.3/69.5 MB 5.1 MB/s eta 0:00:14
   - -------------------------------------- 2.4/69.5 MB 4.6 MB/s eta 0:00:15
   -- ------------------------------------- 3.7/69.5 MB 5.0 MB/s eta 0:00:14
   -- ------------------------------------- 4.5/69.5 MB 5.0 MB/s eta 0:00:13
   --- ------------------------------------ 6.0/69.5 MB 5.4 MB/s eta 0:00:12
   ---- ----------------------------------- 7.6/69.5 MB 5.7 MB/s eta 0:00:11
   ----- ---------------------------------- 9.2/69.5 MB 6.0 MB/s eta 0:00:10
   ------ --------------------------------- 10.5/69.5 MB 5.9 MB/s eta 0:00:10
   ------ --------------------------------- 12.1/69.5 MB 6.1 MB/s eta 0:00:10
   ------- -------------------------------- 13.6/69.5 MB 6.3 MB/s eta 0:00:09
   -------- ------------------------------- 14.9/69.5 MB 6.3 MB/s eta 0:00:09
  

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor

import joblib

In [2]:
df = pd.read_csv("laptops_Featured.csv")
print(df.shape)
df.head(3)

(1014, 15)


,Brand,Processor,GPU,RAM,Storage,Display_Resolution,Refresh_Rate,Price,RAM_GB,Storage_GB,Refresh_Hz,CPU_Score,GPU_Score,Resolution_Score,Performance_Score
0,MSI,Intel Core Ultra 9 285HX,Nvidia GeForce RTX 5090 24GB,96GB DDR5 6400MHz 48GB*2,"6TB, 2TB NVMe PCIe Gen5x4 SSD w/o DRAM + 2TB N...",3840x2400,120 Hz,320000,96,10240,120,10,10.0,5,97.50
1,MSI,Intel Core i9-14900HX,Nvidia GeForce RTX 5070 8GB,16GB DDR5 8GB*2,1TB NVMe PCIe SSD Gen4x4,2560*1440,165 HZ,79999,16,1024,165,10,9.0,3,80.44
2,Acer,Intel Core I5-13420H,Nvidia GeForce RTX 4050 6GB,8 GB DDR5-SDRAM,512 GB SSD,1920x1080,144 Hz,39999,8,512,144,6,7.0,3,55.25


In [3]:
df = df[[
    "Brand",
    "CPU_Score",
    "GPU_Score",
    "RAM_GB",
    "Storage_GB",
    "Refresh_Hz",
    "Resolution_Score",
    "Performance_Score"
]]
df.head()

,Brand,CPU_Score,GPU_Score,RAM_GB,Storage_GB,Refresh_Hz,Resolution_Score,Performance_Score
0,MSI,10,10.0,96,10240,120,5,97.50
1,MSI,10,9.0,16,1024,165,3,80.44
2,Acer,6,7.0,8,512,144,3,55.25
3,Dell,8,8.0,16,512,120,4,67.50
4,Dell,10,8.5,16,1024,120,4,78.75


In [4]:
label_encoders = {}
encoder = LabelEncoder()
df["Brand"] = encoder.fit_transform(df["Brand"])
label_encoders["Brand"] = encoder
df.head()

,Brand,CPU_Score,GPU_Score,RAM_GB,Storage_GB,Refresh_Hz,Resolution_Score,Performance_Score
0,7,10,10.0,96,10240,120,5,97.50
1,7,10,9.0,16,1024,165,3,80.44
2,1,6,7.0,8,512,144,3,55.25
3,2,8,8.0,16,512,120,4,67.50
4,2,10,8.5,16,1024,120,4,78.75


In [5]:
X = df.drop("Performance_Score", axis=1)
y = df["Performance_Score"]
print("X Shape :", X.shape)
print("y Shape :", y.shape)
X.head()

X Shape : (1014, 7)
y Shape : (1014,)


,Brand,CPU_Score,GPU_Score,RAM_GB,Storage_GB,Refresh_Hz,Resolution_Score
0,7,10,10.0,96,10240,120,5
1,7,10,9.0,16,1024,165,3
2,1,6,7.0,8,512,144,3
3,2,8,8.0,16,512,120,4
4,2,10,8.5,16,1024,120,4


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (811, 7)
X_test  : (203, 7)
y_train : (811,)
y_test  : (203,)


In [7]:
model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)
model

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [8]:
print(df["Performance_Score"].isna().sum())

0


In [9]:
df[df["Performance_Score"].isna()]

,Brand,CPU_Score,GPU_Score,RAM_GB,Storage_GB,Refresh_Hz,Resolution_Score,Performance_Score


In [10]:
df.loc[576]

Brand                  2.00
CPU_Score              8.00
GPU_Score              2.00
RAM_GB                16.00
Storage_GB           512.00
Refresh_Hz            60.00
Resolution_Score       3.00
Performance_Score     44.25
Name: 576, dtype: float64

In [11]:
df_domer = pd.read_csv("laptops_Clean.csv")
df_domer.loc[576]

Brand                                                                Dell
Product_Name                                             inspiron 14 7440
Processor                                               Intel Core 7 150U
GPU                                                        Intel Graphics
GPU_Memory                                                            NaN
RAM                                                         2*8 DDR5 5600
Storage                                                         512GB SSD
Display                 14", Touch, FHD+ 1920x1200, 60Hz, WVA, IPS, 25...
Display_Resolution                                            1920 x 1200
Refresh_Rate                                                        60 Hz
Operating_System                                          Windows 11 Home
Keyboard                                                              NaN
Battery                                         4 Cell, 64 Wh, integrated
Webcam                                

In [12]:
print("NaN values in X:")
print(X_train.isna().sum())

print("\nNaN values in y:")
print(y_train.isna().sum())

NaN values in X:
Brand               0
CPU_Score           0
GPU_Score           0
RAM_GB              0
Storage_GB          0
Refresh_Hz          0
Resolution_Score    0
dtype: int64

NaN values in y:
0


In [13]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=10,
    cv=5,
    scoring="r2",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best Parameters:")
print(random_search.best_params_)

print("\nBest Cross Validation Score:")
print(random_search.best_score_)

Best Parameters:
{'subsample': 1.0, 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 1.0}

Best Cross Validation Score:
0.9983801167581289


In [14]:
best_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_estimators=300,
    max_depth=3,
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=1.0
)

best_model.fit(X_train, y_train)

print("Final XGBoost model trained successfully!")

Final XGBoost model trained successfully!


In [15]:
y_pred = best_model.predict(X_test)

print(y_pred[:10])

[25.365788 59.966953 68.37945  42.77753  44.179733 73.053604 46.834183
 60.044407 63.560917 49.846874]


In [16]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R² Score : {r2:.4f}")

MAE  : 0.2570
MSE  : 0.1697
RMSE : 0.4119
R² Score : 0.9993


In [20]:
feature_columns = X.columns.tolist()

artifacts = {
    "model": best_model,
    "label_encoders": label_encoders,
    "feature_columns": feature_columns
}

joblib.dump(artifacts, "model_artifacts.joblib")

print("Model artifacts saved successfully!")

Model artifacts saved successfully!
